# BRCA 10x slide — L2 maps of the finished ablation weights

One slide: `V1_Breast_Cancer_Block_A_Section_1_image` (10x Human Breast Cancer Block A Section 1). The other BRCA slides are not included.

These are the 2026-11-17 weights in `result_uni2_ablation_<arm>_new1066axisfix_paired_v2_seed42`, not the older A2 / A3 / A7 / A8 trees. Each panel is the L2 argmax (`pred_CT`) on the fullres H&E.

Predict first, on a GPU that is not training:

```bash
CUDA_DEVICE=2 bash code/External_val/BRCA/demo_uni2_axisfix.sh predict
CUDA_DEVICE=2 bash code/External_val/predict_fold_ensemble.sh BRCA
```

The notebook reads one prediction file per finished arm. It does not launch a GPU job. The `ens5_*` panels are the mean softmax of the five fold models of one cohort_safe `l12_links_1` seed. They are written by `predict_fold_ensemble.sh`, under `pred_uni2_<name>/`, not `pred_uni2_axisfix_`.

The no-neighbor arms (`legacy_nospatial`, `adjacent_nospatial`, `l12_balanced_nospatial`) are not shown.

In [ ]:
import sys
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D

REPO = Path("/home/lingyu/ssd2/Python/Hist2Pheno")
for _p in (REPO / "code", REPO / "code" / "External_val" / "BRCA"):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

import brca_external as brca_ext

brca_ext.use_embedder("uni2")
cohort = brca_ext.COHORT

SAMPLE = "V1_Breast_Cancer_Block_A_Section_1_image"
BRCA = REPO / "data" / "Visium" / "BRCA"
HE_MAX_SIDE = 1100
DPI = 120
MARKER_PX = 0.7

# Neighbor arms only. legacy_nospatial, adjacent_nospatial, and l12_balanced_nospatial are left out.
ARMS = [
    ("spatial_k8", "trunk: adjacent + k=8 mean"),
    ("spatial_k8_gate", "trunk + spatial gate"),
    ("l12_links_1", "trunk, L12 link weights = 1"),
    # 2026.11.18, released Hist2Pheno = mean of fold models (predict_fold_ensemble.sh BRCA)
    ("ens5_l12_links_1_ivsafe_seed42", "cohort_safe seed 42, mean of 5 folds"),
    ("ens5_l12_links_1_ivsafe_seed44", "cohort_safe seed 44, mean of 5 folds"),
    ("l12_links_2", "trunk, L12 link weights = 2"),
    ("l12_balanced", "trunk + balanced L12"),
    ("focal_gamma1", "trunk + focal γ=1, class weights kept"),
    ("balanced_sampler", "trunk + sampler, no class weight"),
]

# 2026.11.17, PRED_TAG = PROB_TAG of demo_uni2_axisfix.sh ("" for STAR_PROB_MIN=0)
PRED_TAG = ""


ENSEMBLES = {
    "ens5_l12_links_1_ivsafe_seed42",
    "ens5_l12_links_1_ivsafe_seed44",
}


def pred_h5ad(arm):
    # Ensembles are pred_uni2_ens<N>_<run>/, the same names brca_10x_annotation_eval.arm_paths uses.
    folder = f"pred_uni2_{arm}" if arm in ENSEMBLES else f"pred_uni2_axisfix_{arm}"
    return BRCA / f"{folder}{PRED_TAG}" / SAMPLE / f"{SAMPLE}_stardist_pred.h5ad"

In [ ]:
he_path = cohort.sample_paths(SAMPLE)["he"]
thumb, full_w, full_h = cohort._he_thumbnail(he_path, max_side=HE_MAX_SIDE)
palette = cohort._palette()["L2"]
point_s = (MARKER_PX * 72.0 / DPI) ** 2
print(f"{SAMPLE}  HE {he_path.name}  thumb {thumb.shape}  full {full_w} x {full_h}")

loaded = []
missing = []
for arm, note in ARMS:
    path = pred_h5ad(arm)
    if not path.is_file():
        missing.append(arm)
        continue
    adata = ad.read_h5ad(path)
    loaded.append((arm, note, adata))
    print(f"{arm}: {adata.n_obs:,} nuclei  {note}")
ensemble_missing = [arm for arm in missing if arm in ENSEMBLES]
other_missing = [arm for arm in missing if arm not in ENSEMBLES]
if ensemble_missing:
    print(
        "ensemble predictions not written yet: "
        + ", ".join(ensemble_missing)
        + ". Run: CUDA_DEVICE=2 bash code/External_val/predict_fold_ensemble.sh BRCA"
    )
if other_missing:
    raise FileNotFoundError(
        "Missing predictions for "
        + ", ".join(other_missing)
        + ". Run: CUDA_DEVICE=2 bash code/External_val/BRCA/demo_uni2_axisfix.sh predict"
    )


## L2 cell type on the 10x fullres image

The panels share one color scale and one legend. Each point is a StarDist nucleus colored by its L2 argmax.

In [ ]:
by_arm = {arm: adata for arm, note, adata in loaded}
ncols = 3
nrows = (len(ARMS) + 1 + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(16.8, 4.6 * nrows), dpi=DPI)
axes = np.atleast_1d(axes).ravel()
present = set()
for ax, (arm, note) in zip(axes, ARMS):
    adata = by_arm.get(arm)
    if adata is None:
        cohort._scatter_on_he(
            ax, thumb, full_w, full_h, np.zeros((0, 2)), [], palette,
            f"{arm}\nmissing",
            point_s,
            show_he=True,
        )
        continue
    labels = adata.obs["pred_CT"].astype(str).to_numpy()
    xy = np.asarray(adata.obsm["spatial_HE"], dtype=np.float64)
    present |= set(labels.tolist())
    cohort._scatter_on_he(
        ax, thumb, full_w, full_h, xy, labels, palette,
        f"{arm}\n{note}  n={adata.n_obs:,}",
        point_s,
        show_he=True,
    )
for ax in axes[len(ARMS):]:
    ax.set_axis_off()
order = [name for name in palette if name in present]
order += sorted(present - set(order))
handles = [
    Line2D(
        [0], [0], linestyle="none", marker="o", markersize=8,
        markerfacecolor=palette.get(name, (0.5, 0.5, 0.5, 1))[:3],
        markeredgecolor="none", label=name,
    )
    for name in order
]
axes[len(ARMS)].legend(handles=handles, loc="center", frameon=False, fontsize=8, ncol=1)
fig.suptitle("10x Block A Section 1  —  L2 cell type", fontsize=13, y=0.995)
fig.tight_layout(rect=(0, 0, 1, 0.975))
out = BRCA / "pred_uni2_axisfix_figures"
out.mkdir(parents=True, exist_ok=True)
fig.savefig(out / f"{SAMPLE}_L2_axisfix.png", dpi=160, bbox_inches="tight", facecolor="white")
plt.show()

## 2026.11.17 Pathologist spot table, four label modes

`brca_10x_annotation_eval.py` scores every arm against `10x_data/metadata.tsv` (IDC / DCIS/LCIS / Tumor edge / Healthy) with the 55 µm capture disc. Each arm is scored four ways:

- `argmax`: `pred_CT`.
- `decision`: the run's `decision_rule_l2_only.json`.
- `prior_shift`: EM on this slide, with the training prior taken as the mean OOF softmax.
- `smooth`: argmax after kNN smoothing (k=8, α=0.3).

`score` is the mean of three numbers: cancer spots called Tumor/DCIS, healthy spots not called Tumor/DCIS, and the IDC→Tumor / DCIS→DCIS naming rate. The nucleus labels per mode are written next to each prediction as `{SAMPLE}_l2_label_modes.csv`. Set `RADIUS_UM = 65` to reproduce the first table in the PanCancer README.

In [ ]:
import brca_10x_annotation_eval as ev

RADIUS_UM = 55.0
eval_table = ev.evaluate([arm for arm, _ in ARMS], radius_um=RADIUS_UM, pred_tag=PRED_TAG)
cols = ["arm", "mode", "IDC_called_Tumor", "DCIS_called_DCIS", "cancer_called_neoplastic",
        "healthy_not_neoplastic", "name_correct", "score"]
display(eval_table[cols].round(3))
best_mode = (eval_table.groupby("mode")["score"].mean().sort_values(ascending=False))
print("mean score over arms per mode:\n" + best_mode.round(3).to_string())
eval_table.to_csv(BRCA / "pred_uni2_axisfix_figures" / f"{SAMPLE}_annotation_eval_r{RADIUS_UM:g}.csv", index=False)

Maps of one arm in all four modes, so the change the table measures can be seen on the slide.

In [ ]:
import pandas as pd

SHOW_ARM = eval_table.iloc[0]["arm"]          # best (arm, mode) row
modes_csv = pred_h5ad(SHOW_ARM).with_name(f"{SAMPLE}_l2_label_modes.csv")
mode_labels = pd.read_csv(modes_csv)
adata = ad.read_h5ad(pred_h5ad(SHOW_ARM))
if list(mode_labels["obs_name"].astype(str)) != list(adata.obs_names.astype(str)):
    raise RuntimeError("label modes and prediction rows differ")
xy = np.asarray(adata.obsm["spatial_HE"], dtype=np.float64)
mode_cols = [c for c in mode_labels.columns if c != "obs_name"]
fig, axes = plt.subplots(1, len(mode_cols), figsize=(5.6 * len(mode_cols), 5.0), dpi=DPI)
for ax, mode in zip(np.atleast_1d(axes), mode_cols):
    cohort._scatter_on_he(ax, thumb, full_w, full_h, xy, mode_labels[mode].astype(str).to_numpy(),
                          palette, f"{SHOW_ARM}  {mode}", point_s, show_he=True)
fig.tight_layout()
plt.show()